##  Lets List the Files before creating table

In [0]:
%sql
LIST "/Volumes/shopkart/bronze/raw_files/olist"

## Schemas
Explicit schema for orders; other files use inferSchema (see KAN-10).

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

orders_schema = StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_status', StringType(), True), StructField('order_purchase_timestamp', TimestampType(), True), StructField('order_approved_at', TimestampType(), True), StructField('order_delivered_carrier_date', TimestampType(), True), StructField('order_delivered_customer_date', TimestampType(), True), StructField('order_estimated_delivery_date', TimestampType(), True)])

## Ingest every CSV into Bronze
One function loads every table and adds the audit columns `_ingested_at` and `_source_file` (KAN-11).

In [0]:
VOLUME = "/Volumes/shopkart/bronze/raw_files/olist"
SCHEMAS = {"olist_orders_dataset.csv": orders_schema}

def to_table_name(file_name):
    return file_name.replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")

def ingest_csv(file_name, table_name, schema=None):
    reader = spark.read.option("header", True)
    reader = reader.schema(schema) if schema else reader.option("inferSchema", True)
    df = (reader.csv(f"{VOLUME}/{file_name}")
              .withColumns({"_ingested_at": F.current_timestamp(),
                            "_source_file": F.col("_metadata.file_path")}))
    df.write.mode("overwrite").saveAsTable(f"shopkart.bronze.{table_name}")

csv_files = [f.name for f in dbutils.fs.ls(VOLUME) if f.name.endswith(".csv")]
for f in csv_files:
    ingest_csv(f, to_table_name(f), SCHEMAS.get(f))

## Checks
Row counts (KAN-10) and audit columns on every table (KAN-11).

In [0]:
expected = {"orders": 20000, "customers": 20000, "order_items": 23650, "products": 2800, "order_payments": 20314}
for t, n in expected.items():
    actual = spark.table(f"shopkart.bronze.{t}").count()
    print(f"{t:<12} {actual:>7,}  {'OK' if actual == n else 'MISMATCH'}")
    assert actual == n, f"{t}: expected {n}, got {actual}"

In [0]:
%sql
SELECT table_name, count(*) AS audit_cols
FROM shopkart.information_schema.columns
WHERE table_schema = 'bronze' AND column_name IN ('_ingested_at', '_source_file')
GROUP BY table_name ORDER BY table_name;